In [1]:
%pip install -q uv

In [2]:
import sys
!uv pip install --python "{sys.executable}" langchain langchain-groq

Using Python 3.13.18 environment at: /usr
Checked 2 packages in 62ms


In [3]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
print("Groq API key loaded.")

Groq API key loaded.


In [5]:
from langchain_groq import ChatGroq

# Initialize the Groq model.
llm = ChatGroq(
   model="openai/gpt-oss-20b"
)

# Create a simple customer support question.
messages = [
    (
        "system",
        "You are a customer support representative. "
        "Your goal is to efficiently handle queries."
    ),
    ("human", "What are your business hours?"),
]

# Send the messages and display the response.
result = llm.invoke(messages)
print(result.content)

Our customer support team is available Monday through Friday from **9 AM to 6 PM** (local time). If you need assistance outside those hours, feel free to leave a message or submit a ticket, and we’ll get back to you during the next business day.


In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq

# Initialize LangChain with the Groq model.
llm = ChatGroq(
    model="openai/gpt-oss-20b"
)

# Step 1: Create the first prompt template to retrieve customer query details.
first_prompt_template = PromptTemplate.from_template(
    "Extract the key information from the following customer query: "
    "{customer_query}"
)

# Step 2: Create the second prompt template to fetch order details
# based on the extracted information.
second_prompt_template = PromptTemplate.from_template(
    "Using the extracted information: {extracted_info}, "
    "retrieve the relevant order details."
)

# Step 3: Create the third prompt template to generate a detailed
# response for the customer.
third_prompt_template = PromptTemplate.from_template(
    "Based on the following order details: {order_details}, "
    "generate a detailed response to address the customer's query."
)

# Step 4: Define the chains using current LangChain syntax.
# StrOutputParser converts each model response into text.
first_chain = first_prompt_template | llm | StrOutputParser()
second_chain = second_prompt_template | llm | StrOutputParser()
third_chain = third_prompt_template | llm | StrOutputParser()

# Input to start the prompt chain: an example customer query.
initial_input = (
    "Can you help me with the status of my order #12345 placed last week?"
)

# Run the first chain to extract key information from the query.
extracted_info = first_chain.invoke({"customer_query": initial_input})
print("Extracted Info:", extracted_info)

# Run the second chain to request order details based on the extracted information.
order_details = second_chain.invoke({"extracted_info": extracted_info})
print("\nOrder Details:", order_details)

# Run the third chain to generate a detailed response based on the order details.
customer_response = third_chain.invoke({"order_details": order_details})
print("\nCustomer Response:", customer_response)

Extracted Info: **Key Information Extracted**

- **Order ID:** 12345  
- **Order Placement Timing:** Last week (exact date not specified)  
- **Requested Action:** Check the current status of the order.

Order Details: I’m sorry, but I don’t have direct access to your order database or the shipping system, so I can’t pull up the live status of order 12345 for you.  

Here’s what you can do next:

| Option | How to do it | What to look for |
|--------|--------------|------------------|
| **Check the order confirmation email** | Open the email you received when you placed the order. | Look for a tracking number or a link that says “View Order Status.” |
| **Log in to the retailer’s website** | Go to the site’s “My Orders” or “Order History” section. | Find order 12345 and see the current status (e.g., “Processing,” “Shipped,” “Out for delivery,” etc.). |
| **Use the mobile app** | If the retailer has an app, open it and navigate to your orders. | Same as above – you’ll see the status and

In [11]:
# Improved Prompt 1
first_prompt_template = PromptTemplate.from_template(
    "Extract the order number and customer request from: {customer_query}. "
    "If a detail is missing, say 'not provided'."
)

# Improved Prompt 2
second_prompt_template = PromptTemplate.from_template(
    "Using only the information in {extracted_info}, list the "
    "information needed to check the order status. Do not give "
    "example names, addresses, dates, payment details, or order statuses."
)

# Improved Prompt 3
third_prompt_template = PromptTemplate.from_template(
    "Using only {order_details}, write a short customer response. "
    "Do not address the customer by name or include example personal "
    "information. State that the actual order status must be verified."
)
# Create the chains using the improved prompts.
first_chain = first_prompt_template | llm | StrOutputParser()
second_chain = second_prompt_template | llm | StrOutputParser()
third_chain = third_prompt_template | llm | StrOutputParser()

# Test with a customer query.
initial_input = "Can you help me with the status of my order #12345 placed last week?"

# Run the first chain.
extracted_info = first_chain.invoke({"customer_query": initial_input})
print("Extracted Info:", extracted_info)

# Run the second chain.
order_details = second_chain.invoke({"extracted_info": extracted_info})
print("\nOrder Details Needed:", order_details)

# Run the third chain.
customer_response = third_chain.invoke({"order_details": order_details})
print("\nCustomer Response:", customer_response)


Extracted Info: Order number: **12345**  
Customer request: **status of my order**

Order Details Needed: **Information required to check an order status**

- Order number  
- Customer email address (or account ID)  
- Phone number used during checkout  
- Shipping address (street, city, state/province, postal code)  
- Billing address (if different from shipping)  
- Payment method identifier (e.g., last four digits of card, PayPal ID)  

These are the typical pieces of data that a support agent would request in order to locate and verify the status of an order.

Customer Response: To locate and verify your order, please provide the following details:

- Order number  
- Customer email address or account ID  
- Phone number used at checkout  
- Shipping address (street, city, state/province, postal code)  
- Billing address (if different from shipping)  
- Payment method identifier (e.g., last four digits of card, PayPal ID)

Once we have this information, we can verify the actual ord